# Python `functools` — Complete Notes (DSA Edition)

## Table of Contents
1. Why functools?
2. @lru_cache — Memoization (MOST IMPORTANT FOR DSA)
3. @cache (Python 3.9+)
4. functools.reduce()
5. functools.partial()
6. functools.cmp_to_key() — Custom Sorting (IMPORTANT FOR DSA)
7. functools.wraps() — Decorator Helper
8. functools.total_ordering
9. Strings Deep Dive with functools
10. Lists Deep Dive with functools
11. Dictionaries with functools
12. Classic DSA Problems solved with functools
13. Cheat Sheet
14. Exercises


## Section 1: Why functools?

`functools` is a standard Python library module that provides higher-order functions (functions that act on or return other functions).
It solves problems related to performance (memoization), code reuse (partial evaluation), custom sorting, and decorator metadata preservation.

### Overview Table

| Tool | Purpose | DSA Use Case |
|---|---|---|
| `@lru_cache` / `@cache` | Memoization (caching return values of a function) | Dynamic Programming (Top-down DP) |
| `cmp_to_key` | Converts a comparison function to a key function | Custom sorting (e.g., Largest Number problem) |
| `reduce` | Applies function of two arguments cumulatively to a sequence | Accumulation, aggregate calculations |
| `partial` | Returns a new function with partial application of arguments | Creating specialized helper functions |
| `wraps` | Preserves metadata of decorated function | Writing custom decorators (timing, debug) |
| `@total_ordering` | Generates rich comparison methods given `__eq__` and one other | Custom Node classes for Heaps/Trees |


## Section 2: @lru_cache — Memoization (MOST IMPORTANT FOR DSA)

Memoization is an optimization technique used primarily to speed up computer programs by storing the results of expensive function calls and returning the cached result when the same inputs occur again.
Naive recursion is often extremely slow (exponential time complexity, $O(2^N)$) because it recomputes the same subproblems repeatedly. Memoization reduces this to polynomial time (often $O(N)$).


### Fibonacci WITHOUT memoization
This is very slow for larger numbers.


In [ ]:
import time

def fib_naive(n):
    if n <= 1:
        return n
    return fib_naive(n-1) + fib_naive(n-2)

start = time.time()
print("Fib 35 naive:", fib_naive(35))
print(f"Time taken (naive): {time.time() - start:.4f} seconds")


### Fibonacci WITH @lru_cache
Notice how fast it becomes by caching previous calls!


In [ ]:
from functools import lru_cache
import time

@lru_cache(maxsize=None)
def fib_memo(n):
    if n <= 1:
        return n
    return fib_memo(n-1) + fib_memo(n-2)

start = time.time()
print("Fib 35 memoized:", fib_memo(35))
print(f"Time taken (memo): {time.time() - start:.6f} seconds")


### Visualize call tree (ascii diagram)
Let's see how many calls are skipped by the cache.


In [ ]:
print("Naive Fibonacci(4) call tree:")
print("fib(4)")
print("├── fib(3)")
print("│   ├── fib(2)")
print("│   │   ├── fib(1)")
print("│   │   └── fib(0)")
print("│   └── fib(1)")
print("└── fib(2)")
print("    ├── fib(1)")
print("    └── fib(0)")
print("\nMemoized Fibonacci(4) call tree:")
print("fib(4)")
print("├── fib(3)")
print("│   ├── fib(2)")
print("│   │   ├── fib(1)")
print("│   │   └── fib(0)")
print("│   └── fib(1) (CACHED)")
print("└── fib(2) (CACHED)")


### @lru_cache(maxsize=None) vs @lru_cache(maxsize=128)
`maxsize=None` means the cache can grow indefinitely (good for DP if states are limited).
`maxsize=128` (or any integer) means only the 128 most recent unique calls are cached (Least Recently Used replacement).


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=2)
def limited_cache(x):
    print(f"Computing {x}...")
    return x * x

limited_cache(1)
limited_cache(2)
limited_cache(3) # Evicts '1'
print("Calling 1 again:")
limited_cache(1) # Recomputes because it was evicted!


### cache_info() — see hits, misses, maxsize, currsize


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def factorial(n):
    return n * factorial(n-1) if n else 1

factorial(5)
factorial(7) # 5 is cached, only computes 6 and 7
print(factorial.cache_info())


### cache_clear() — clear the cache


In [ ]:
factorial.cache_clear()
print(factorial.cache_info()) # Cache is now empty


### DSA — Climbing Stairs problem (LeetCode style) with @lru_cache
You are climbing a staircase. It takes n steps to reach the top. Each time you can either climb 1 or 2 steps. In how many distinct ways can you climb to the top?


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def climb_stairs(n):
    if n == 1: return 1
    if n == 2: return 2
    return climb_stairs(n-1) + climb_stairs(n-2)

print("Ways to climb 10 stairs:", climb_stairs(10))


### DSA — Coin Change problem with @lru_cache


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def coin_change(amount, coins_tuple): # Note: args must be hashable, so list -> tuple
    if amount == 0: return 0
    if amount < 0: return float('inf')
    
    res = float('inf')
    for coin in coins_tuple:
        res = min(res, 1 + coin_change(amount - coin, coins_tuple))
    return res

coins = (1, 2, 5)
amount = 11
ans = coin_change(amount, coins)
print(f"Min coins to make {amount} with {coins}: {ans}")


### DSA — Longest Common Subsequence (LCS) with @lru_cache


In [ ]:
from functools import lru_cache

def lcs(text1, text2):
    @lru_cache(maxsize=None)
    def dp(i, j):
        if i == len(text1) or j == len(text2):
            return 0
        if text1[i] == text2[j]:
            return 1 + dp(i+1, j+1)
        return max(dp(i+1, j), dp(i, j+1))
    
    return dp(0, 0)

print("LCS of 'abcde' and 'ace':", lcs("abcde", "ace"))


### DSA — 0/1 Knapsack with @lru_cache


In [ ]:
from functools import lru_cache

def knapsack(weights, values, capacity):
    # Using tuples since lists are unhashable
    w_tuple = tuple(weights)
    v_tuple = tuple(values)
    
    @lru_cache(maxsize=None)
    def dp(i, curr_capacity):
        if i == len(w_tuple) or curr_capacity == 0:
            return 0
        
        # Skip item if it's too heavy
        if w_tuple[i] > curr_capacity:
            return dp(i+1, curr_capacity)
            
        # Take or don't take
        take = v_tuple[i] + dp(i+1, curr_capacity - w_tuple[i])
        skip = dp(i+1, curr_capacity)
        return max(take, skip)
        
    return dp(0, capacity)

w = [1, 3, 4, 5]
v = [1, 4, 5, 7]
print("Knapsack max value:", knapsack(w, v, 7))


### DSA — Grid path counting (unique paths) with @lru_cache


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def unique_paths(m, n):
    if m == 1 or n == 1:
        return 1
    return unique_paths(m-1, n) + unique_paths(m, n-1)

print("Unique paths in 3x7 grid:", unique_paths(3, 7))


### Matplotlib — bar chart showing recursive calls with/without memoization


In [ ]:
import matplotlib.pyplot as plt

# Theoretical call counts for Fibonacci
n_vals = [5, 10, 15, 20]
naive_calls = [15, 177, 1973, 21891]
memo_calls = [9, 19, 29, 39]

plt.figure(figsize=(10, 5))
plt.bar([str(n) for n in n_vals], naive_calls, label='Naive Calls', alpha=0.7)
plt.bar([str(n) for n in n_vals], memo_calls, label='Memoized Calls', alpha=0.9)
plt.yscale('log')
plt.xlabel('N')
plt.ylabel('Number of Function Calls (Log Scale)')
plt.title('Recursive Calls: Naive vs Memoized Fibonacci')
plt.legend()
plt.show()


## Section 3: @cache (Python 3.9+)

`@cache` is simply a more readable alias for `@lru_cache(maxsize=None)`.


### @cache usage


In [ ]:
from functools import cache

@cache
def fast_fib(n):
    if n <= 1: return n
    return fast_fib(n-1) + fast_fib(n-2)

print("Fib 50 with @cache:", fast_fib(50))


### When to use @cache vs @lru_cache


In [ ]:
# Use @cache when you want to memoize EVERYTHING (e.g. DP problems where state space is bounded)
# Use @lru_cache(maxsize=128) when state space is infinite/huge and you only want to cache recent items to save memory.
print("Rule of thumb for LeetCode: Just use @cache (or @lru_cache(None) for older Python)")


### DSA — nth Tribonacci number with @cache


In [ ]:
from functools import cache

@cache
def tribonacci(n):
    if n == 0: return 0
    if n <= 2: return 1
    return tribonacci(n-1) + tribonacci(n-2) + tribonacci(n-3)

print("Tribonacci 25:", tribonacci(25))


## Section 4: functools.reduce()

`reduce(function, iterable[, initializer])` applies a function of two arguments cumulatively to the items of a sequence, from left to right, so as to reduce the sequence to a single value.


### reduce() to sum a list


In [ ]:
from functools import reduce

nums = [1, 2, 3, 4, 5]
# (((1+2)+3)+4)+5
total = reduce(lambda x, y: x + y, nums)
print("Sum:", total)


### reduce() to find maximum of a list


In [ ]:
from functools import reduce

nums = [1, 2, 3, 4, 5]
max_val = reduce(lambda a, b: a if a > b else b, nums)
print("Max:", max_val)


### reduce() to find product of all elements


In [ ]:
from functools import reduce

nums = [1, 2, 3, 4, 5]
product = reduce(lambda x, y: x * y, nums)
print("Product:", product)


### reduce() to flatten a list of lists


In [ ]:
from functools import reduce

nested = [[1, 2], [3, 4], [5]]
flat = reduce(lambda a, b: a + b, nested)
print("Flattened:", flat)


### reduce() to build a dictionary from a list of tuples


In [ ]:
from functools import reduce

pairs = [('a', 1), ('b', 2), ('c', 3)]
def add_to_dict(d, pair):
    d[pair[0]] = pair[1]
    return d

res_dict = reduce(add_to_dict, pairs, {})
print("Built dictionary:", res_dict)


### DSA — compute factorial using reduce


In [ ]:
from functools import reduce

n = 5
fact = reduce(lambda x, y: x * y, range(1, n+1))
print("Factorial of 5:", fact)


### DSA — compute GCD of a list of numbers using reduce


In [ ]:
from functools import reduce
import math

arr = [24, 36, 48, 60]
list_gcd = reduce(math.gcd, arr)
print("GCD of list:", list_gcd)


### DSA — find longest string in a list using reduce


In [ ]:
from functools import reduce

words = ["python", "functools", "reduce", "algorithms"]
longest = reduce(lambda a, b: a if len(a) > len(b) else b, words)
print("Longest word:", longest)


### Comparison: reduce() vs built-in functions


In [ ]:
# sum(nums) is better than reduce(lambda x, y: x + y, nums)
# max(nums) is better than reduce(...)
# math.prod(nums) (Python 3.8+) is better than reduce for product
# reduce is best for custom accumulation logic (like math.gcd over a list)
print("Use built-ins when available, fallback to reduce for custom logic.")


## Section 5: functools.partial()

`partial` lets you freeze some arguments of a function, creating a new object with a simplified signature.


### Basic partial — fix first argument


In [ ]:
from functools import partial

def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube = partial(power, exp=3)

print("Square of 4:", square(base=4))


### partial with keyword arguments


In [ ]:
from functools import partial

def greet(greeting, name, punctuation="."):
    return f"{greeting}, {name}{punctuation}"

shout_greet = partial(greet, punctuation="!!!")
print(shout_greet("Hello", "Alice"))


### partial with built-in functions


In [ ]:
from functools import partial

base2 = partial(int, base=2)
print("1010 in base 2 is:", base2("1010"))


### partial for sorting


In [ ]:
from functools import partial

sort_desc = partial(sorted, reverse=True)
print("Sorted desc:", sort_desc([1, 5, 2, 9, 3]))


### DSA — create specialized search functions using partial


In [ ]:
from functools import partial

def binary_search(arr, target, left, right):
    if left > right: return -1
    mid = (left + right) // 2
    if arr[mid] == target: return mid
    elif arr[mid] < target: return binary_search(arr, target, mid+1, right)
    else: return binary_search(arr, target, left, mid-1)

arr = [1, 2, 3, 4, 5, 6, 7]
# Create a specialized search for this specific array
search_arr = partial(binary_search, arr, left=0, right=len(arr)-1)
print("Found 5 at index:", search_arr(target=5))


### Real use — create a custom print function with prefix


In [ ]:
from functools import partial
import sys

error_print = partial(print, "ERROR:", file=sys.stderr)
error_print("This is a custom error message")


### Real use — create multiply_by_2, multiply_by_3


In [ ]:
from functools import partial

def multiply(a, b):
    return a * b

mul_2 = partial(multiply, 2)
mul_3 = partial(multiply, 3)

print("10 * 2 =", mul_2(10))
print("10 * 3 =", mul_3(10))


## Section 6: functools.cmp_to_key() — Custom Sorting (IMPORTANT FOR DSA)

Python 3 removed the `cmp` argument from sorting functions. `cmp_to_key` converts an old-style comparison function to a key function.
A comparison function returns:
- negative value if a < b
- zero if a == b
- positive value if a > b


### Basic cmp_to_key usage


In [ ]:
from functools import cmp_to_key

def compare(a, b):
    # Sort ascending
    if a < b: return -1
    elif a > b: return 1
    return 0

nums = [4, 2, 5, 1, 3]
print("Sorted:", sorted(nums, key=cmp_to_key(compare)))


### DSA — Largest Number problem
Given [3, 30, 34, 5, 9], form the largest string "9534330".


In [ ]:
from functools import cmp_to_key

def compare_strings(a, b):
    if a + b > b + a:
        return -1 # a comes first
    elif a + b < b + a:
        return 1
    return 0

nums = [3, 30, 34, 5, 9]
str_nums = list(map(str, nums))
str_nums.sort(key=cmp_to_key(compare_strings))
largest = "".join(str_nums)
print("Largest number:", largest)


### DSA — Sort by multiple criteria with custom comparator
Sort points (x,y): by x ascending, if equal, by y descending.


In [ ]:
from functools import cmp_to_key

def compare_points(p1, p2):
    if p1[0] != p2[0]:
        return -1 if p1[0] < p2[0] else 1
    else:
        return -1 if p1[1] > p2[1] else 1 # y descending

points = [(1, 5), (2, 3), (1, 10), (2, 8)]
points.sort(key=cmp_to_key(compare_points))
print("Sorted points:", points)


### DSA — Sort fractions without converting to float
Avoid floating point precision issues by cross-multiplying.


In [ ]:
from functools import cmp_to_key

def compare_fractions(f1, f2):
    # f = (num, den). Compare f1_num/f1_den vs f2_num/f2_den
    # Equivalent to comparing f1_num * f2_den vs f2_num * f1_den
    v1 = f1[0] * f2[1]
    v2 = f2[0] * f1[1]
    return v1 - v2 # Ascending

fracs = [(1, 3), (1, 2), (2, 5)]
fracs.sort(key=cmp_to_key(compare_fractions))
print("Sorted fractions:", fracs)


### DSA — Sort strings by frequency then alphabetically


In [ ]:
from functools import cmp_to_key
from collections import Counter

words = ["apple", "banana", "apple", "cherry", "cherry", "cherry"]
freq = Counter(words)

def compare_words(w1, w2):
    if freq[w1] != freq[w2]:
        return freq[w2] - freq[w1] # Descending freq
    if w1 < w2:
        return -1 # Ascending alphabetical
    return 1

unique_words = list(freq.keys())
unique_words.sort(key=cmp_to_key(compare_words))
print("Sorted words:", unique_words)


### Comparison: cmp_to_key vs key= lambda


In [ ]:
# key=lambda is better for simple property access (e.g. key=lambda x: x[1])
# cmp_to_key is REQUIRED when comparison depends on BOTH elements simultaneously
# (like the Largest Number problem where a+b > b+a)
print("Use cmp_to_key only when the relation depends on both elements combined.")


## Section 7: functools.wraps() — Decorator Helper

When writing custom decorators, the original function's name and docstring are lost. `@wraps` restores them.


### Decorator WITHOUT wraps


In [ ]:
def my_decorator_no_wraps(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator_no_wraps
def dummy_func():
    """This is a dummy docstring."""
    pass

print("Name:", dummy_func.__name__) # 'wrapper'
print("Doc:", dummy_func.__doc__)   # None


### Decorator WITH @wraps


In [ ]:
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def dummy_func2():
    """This is a dummy docstring."""
    pass

print("Name:", dummy_func2.__name__) # 'dummy_func2'
print("Doc:", dummy_func2.__doc__)   # 'This is a dummy docstring.'


### Write a timing decorator with @wraps


In [ ]:
from functools import wraps
import time

def timeit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        res = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.4f}s")
        return res
    return wrapper

@timeit
def slow_func():
    time.sleep(0.1)
    
slow_func()


### Write a retry decorator with @wraps


In [ ]:
from functools import wraps

def retry(times=3):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if i == times - 1:
                        raise e
                    print(f"Retrying {func.__name__}...")
        return wrapper
    return decorator

@retry(2)
def unstable():
    print("Failing!")
    raise ValueError("Oops")

try: unstable()
except: pass


### Write a memoization decorator from scratch using @wraps


In [ ]:
from functools import wraps

def my_memoize(func):
    cache = {}
    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper

@my_memoize
def fib_custom(n):
    if n <= 1: return n
    return fib_custom(n-1) + fib_custom(n-2)

print("Fib 40 custom memo:", fib_custom(40))


## Section 8: functools.total_ordering

If you define `__eq__` and one of `__lt__`, `__le__`, `__gt__`, or `__ge__`, `@total_ordering` supplies the rest.


### Class without total_ordering


In [ ]:
class NodeNoOrder:
    def __init__(self, val):
        self.val = val
    def __eq__(self, other): return self.val == other.val
    def __lt__(self, other): return self.val < other.val
    # Have to write __le__, __gt__, __ge__ manually!


### Class WITH @total_ordering


In [ ]:
from functools import total_ordering

@total_ordering
class Node:
    def __init__(self, val):
        self.val = val
    def __eq__(self, other):
        return self.val == other.val
    def __lt__(self, other):
        return self.val < other.val

n1, n2 = Node(5), Node(10)
print("n1 <= n2:", n1 <= n2) # Magically works
print("n1 > n2:", n1 > n2)   # Magically works


### DSA — Comparable node class for priority queue


In [ ]:
from functools import total_ordering
import heapq

@total_ordering
class PQNode:
    def __init__(self, weight, vertex):
        self.weight = weight
        self.vertex = vertex
        
    def __eq__(self, other):
        return self.weight == other.weight
        
    def __lt__(self, other):
        return self.weight < other.weight

pq = []
heapq.heappush(pq, PQNode(10, 'A'))
heapq.heappush(pq, PQNode(5, 'B'))
print("Popped vertex with min weight:", heapq.heappop(pq).vertex)


## Section 9: Strings Deep Dive with functools


### Use reduce() to count character frequencies


In [ ]:
from functools import reduce

def count_char(acc, char):
    acc[char] = acc.get(char, 0) + 1
    return acc

s = "functools"
freq = reduce(count_char, s, {})
print("Frequencies:", freq)


### Use lru_cache for recursive palindrome check


In [ ]:
from functools import cache

@cache
def is_palindrome(s, left, right):
    if left >= right: return True
    if s[left] != s[right]: return False
    return is_palindrome(s, left+1, right-1)

word = "racecar"
print("Is palindrome:", is_palindrome(word, 0, len(word)-1))


### Use lru_cache for longest palindromic subsequence


In [ ]:
from functools import cache

def lps(s):
    @cache
    def dp(l, r):
        if l > r: return 0
        if l == r: return 1
        if s[l] == s[r]:
            return 2 + dp(l+1, r-1)
        return max(dp(l+1, r), dp(l, r-1))
    return dp(0, len(s)-1)

print("LPS of 'bbbab':", lps("bbbab"))


### Use cmp_to_key to sort strings by length first, then alphabetically


In [ ]:
from functools import cmp_to_key

def custom_str_cmp(s1, s2):
    if len(s1) != len(s2):
        return len(s1) - len(s2)
    return -1 if s1 < s2 else (1 if s1 > s2 else 0)

arr = ["banana", "apple", "cat", "dog"]
print("Sorted strings:", sorted(arr, key=cmp_to_key(custom_str_cmp)))


## Section 10: Lists Deep Dive with functools


### reduce() to find max subarray sum (Kadane's algorithm style)


In [ ]:
from functools import reduce

nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
# state = (current_max, global_max)
def kadane_step(state, x):
    curr, glob = state
    curr = max(x, curr + x)
    glob = max(glob, curr)
    return (curr, glob)

_, max_sum = reduce(kadane_step, nums, (0, -float('inf')))
print("Max subarray sum:", max_sum)


### lru_cache for recursive merge sort


In [ ]:
from functools import cache

@cache
def merge_sort_tuple(tup):
    if len(tup) <= 1: return tup
    mid = len(tup) // 2
    left = merge_sort_tuple(tup[:mid])
    right = merge_sort_tuple(tup[mid:])
    
    res = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] < right[j]:
            res.append(left[i]); i+=1
        else:
            res.append(right[j]); j+=1
    res.extend(left[i:])
    res.extend(right[j:])
    return tuple(res)

print("Merge sorted:", merge_sort_tuple((5, 3, 8, 1, 2)))


### partial to create specialized filter functions for lists


In [ ]:
from functools import partial

def is_multiple(n, mult):
    return n % mult == 0

is_even = partial(is_multiple, mult=2)
is_div_by_5 = partial(is_multiple, mult=5)

nums = [1, 2, 3, 4, 5, 10, 15]
print("Evens:", list(filter(is_even, nums)))
print("Div by 5:", list(filter(is_div_by_5, nums)))


### cmp_to_key to sort list of lists


In [ ]:
from functools import cmp_to_key

intervals = [[1, 3], [2, 6], [8, 10], [15, 18], [2, 4]]
def cmp_intervals(a, b):
    if a[0] != b[0]: return a[0] - b[0]
    return a[1] - b[1]

intervals.sort(key=cmp_to_key(cmp_intervals))
print("Sorted intervals:", intervals)


## Section 11: Dictionaries with functools


### reduce() to merge multiple dicts


In [ ]:
from functools import reduce

dicts = [{'a': 1}, {'b': 2}, {'c': 3, 'a': 10}]
def merge_dicts(d1, d2):
    res = d1.copy()
    res.update(d2)
    return res

print("Merged:", reduce(merge_dicts, dicts))


### reduce() to compute nested dict value


In [ ]:
from functools import reduce

nested_dict = {'user': {'profile': {'age': 25}}}
path = ['user', 'profile', 'age']

val = reduce(lambda d, key: d[key], path, nested_dict)
print("Nested value:", val)


### lru_cache on function that takes frozenset (hashable dict alternative)
Dicts can't be hashed. If you need a state represented by a dict in DP, use frozenset.


In [ ]:
from functools import cache

@cache
def process_state(fs):
    return len(fs)

d = {'a': 1, 'b': 2}
fs = frozenset(d.items())
print("Processed state:", process_state(fs))


## Section 12: Classic DSA Problems solved with functools


### House Robber (DP) — with @lru_cache


In [ ]:
from functools import cache

def rob(nums):
    nums_tup = tuple(nums)
    @cache
    def dp(i):
        if i >= len(nums_tup): return 0
        return max(dp(i+1), nums_tup[i] + dp(i+2))
    return dp(0)

print("Max money robbed:", rob([2, 7, 9, 3, 1]))


### Word Break problem — with @lru_cache


In [ ]:
from functools import cache

def word_break(s, wordDict):
    wordSet = frozenset(wordDict)
    @cache
    def dp(i):
        if i == len(s): return True
        for j in range(i+1, len(s)+1):
            if s[i:j] in wordSet and dp(j):
                return True
        return False
    return dp(0)

print("Can break 'leetcode':", word_break("leetcode", ["leet", "code"]))


### Sort by custom comparator — Interview problem with cmp_to_key
Sort array by frequency, then value descending.


In [ ]:
from functools import cmp_to_key
from collections import Counter

def freq_sort(nums):
    freq = Counter(nums)
    def cmp(a, b):
        if freq[a] != freq[b]: return freq[a] - freq[b] # Ascending freq
        return b - a # Descending value
    return sorted(nums, key=cmp_to_key(cmp))

print("Custom sorted:", freq_sort([1, 1, 2, 2, 2, 3]))


### Decorator for measuring function call count
Useful for analyzing recursion complexity.


In [ ]:
from functools import wraps

def call_counter(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@call_counter
def test_func():
    pass

test_func()
test_func()
print("Calls:", test_func.calls)


## Section 13: Cheat Sheet

- `@lru_cache(maxsize=None)` / `@cache`: Add to any recursive DP function to make it fast. Arguments must be hashable.
- `cmp_to_key(func)`: Use in `sort(key=...)` when comparison logic relies on interaction between `a` and `b` (like Largest Number).
- `reduce(func, seq, init)`: Use for accumulating a single result from an iterable. (Built-ins like `sum()` are faster if applicable).
- `partial(func, arg=...)`: Pre-fill arguments to create a simpler function.
- `@wraps(func)`: ALWAYS use this when writing custom decorators to keep docstrings and names.
- `@total_ordering`: Automatically generate missing comparison operators for custom classes.


## Section 14: Exercises

Try to solve these without looking at the solutions first!


### Ex1: Fibonacci with memoization using @cache (Try here)


In [ ]:
# Write your solution here


#### Solution


In [ ]:
from functools import cache

@cache
def fib(n):
    if n <= 1: return n
    return fib(n-1) + fib(n-2)


### Ex2: Implement factorial using reduce() (Try here)


In [ ]:
# Write your solution here


#### Solution


In [ ]:
from functools import reduce

def factorial(n):
    if n == 0: return 1
    return reduce(lambda x, y: x * y, range(1, n+1))


### Ex3: Largest Number problem using cmp_to_key (Try here)


In [ ]:
# Write your solution here


#### Solution


In [ ]:
from functools import cmp_to_key

def largestNumber(nums):
    def cmp(a, b):
        if a+b > b+a: return -1
        return 1
    strs = list(map(str, nums))
    strs.sort(key=cmp_to_key(cmp))
    return "".join(strs) if strs[0] != '0' else '0'


### Ex4: Write a debug decorator using @wraps (Try here)


In [ ]:
# Write your solution here


#### Solution


In [ ]:
from functools import wraps

def debug(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with {args} {kwargs}")
        return func(*args, **kwargs)
    return wrapper


### Ex5: Coin change problem (minimum coins) using @lru_cache (Try here)


In [ ]:
# Write your solution here


#### Solution


In [ ]:
from functools import lru_cache

def coinChange(coins, amount):
    coins_tuple = tuple(coins)
    
    @lru_cache(None)
    def dp(amt):
        if amt == 0: return 0
        if amt < 0: return float('inf')
        return min((1 + dp(amt - c) for c in coins_tuple), default=float('inf'))
        
    res = dp(amount)
    return res if res != float('inf') else -1
